In [135]:
from utils.utils import load_from_pickle

testset = load_from_pickle('testset')
dataset = testset.to_pandas()

In [136]:
dataset2 = dataset[:30]

In [137]:
dataset2.head(1)

,question,contexts,ground_truth,evolution_type,metadata,episode_done
0,What challenges does the concept of kearifan l...,[DOC NAME: Undang-undang Nomor 15 Tahun 2023 T...,The concept of kearifan lokal in Bali faces ch...,simple,[{'title': 'Undang-undang Nomor 15 Tahun 2023 ...,True


In [138]:
questions = list(dataset2["question"])
# questions

In [139]:
from utils.llm import get_llm, get_embedding_model
from utils.database.qdrant import QdrantIndexManager
from utils.retriever.uu.embedding_uu import get_embedding_uu
from utils.retriever.uu.graph_uu import get_graph_uu
from utils.config import get_config
from utils.chain.rag import create_combined_answer_chain


llm = get_llm("nvidia-llama3.1")
embed_model = get_embedding_model("BAAI/bge-m3")
vector_store = QdrantIndexManager(embed_model=embed_model, index_name="uu").load_vector_index()
retriever_uu = get_embedding_uu(vector_store=vector_store, llm=llm, embed_model=embed_model, with_self_query=False)

config = get_config()
graph_chain = get_graph_uu(llm=llm, config=config)

chain_quality = create_combined_answer_chain(
  retriever_uu=retriever_uu,
  graph_chain=graph_chain,
  llm=llm
)

Loaded index 'uu'.


In [140]:
result = chain_quality.invoke({"question": "Apa ibukota dari Provinsi Jawa Barat?", "chat_history": ""})
result

{'rewrited question': 'Apa ibukota dari Provinsi Jawa Barat?',
 'context': [Document(metadata={'title': 'Undang-undang Nomor 10 Tahun 2023 Tentang Provinsi Jawa Barat', 'jenis_bentuk_peraturan': 'UNDANG-UNDANG', 'pemrakarsa': 'PEMERINTAH PUSAT', 'nomor': 10, 'tahun': 2023, 'tentang': 'PROVINSI JAWA BARAT', 'tempat_penetapan': 'Jakarta', 'ditetapkan_tanggal': '04 Mei 2023', 'status': 'Berlaku', 'dokumen_peraturan': 'https://peraturan.go.id/files/uu-no-10-tahun-2023.pdf', 'id': 'hqel0eiu', 'filename': 'uu_04_mei_2023_uu_no_10_tahun_2023_hqel0eiu.pdf', 'page_number': 3, '_id': '007eea41-daf7-4a32-ad0f-7ef0ea342534', '_collection_name': 'uu'}, page_content='- 3 -\nBAB II\nCAKUPAN WILAYAH, IBU KOTA, DAN KARAKTERISTIK\nPROVINSI JAWA BARAT\nPasal 3\nProvinsi Jawa Barat terdiri atas 18 (delapan belas).\nkabupaten dan 9 (sembilan) kota, yaitu:.\n Kabupaten Bogor;\nb.\nKabupaten Sukabumi;\nKabupaten Cianjur;\nd. Kabupaten Bandung;\ne.\nKabupaten Garut;\nf.\nKabupaten Tasikmalaya;\nKabupaten Ciam

In [142]:
# def unique_context(context: list):
#     context_unique = [i for n, i in enumerate(context) if i not in context[:n]]
#     return context_unique
from langchain_core.prompt_values import StringPromptValue
from langchain_core.documents.base import Document

def unique_context(context: list):
    context_unique = []
    
    for ctx in context:
        # Handle cases where context is a list of multiple documents/values
        if isinstance(ctx, list):
            unique_items = []
            for item in ctx:
                if isinstance(item, StringPromptValue):
                    content = item.text
                elif isinstance(item, Document):
                    content = item.page_content
                else:
                    continue
                
                # Add unique content only
                if content not in unique_items:
                    unique_items.append(content)
            context_unique.append(unique_items)
        else:
            # If it's a single context, process as a single string or document
            if isinstance(ctx, StringPromptValue):
                context_unique.append(ctx.text)
            elif isinstance(ctx, Document):
                context_unique.append(ctx.page_content)
            else:
                context_unique.append(ctx)  # In case there are other types

    return context_unique

# Apply this function to your context list for each question
ctx_unique = [unique_context(ctx) for ctx in context]

# Now the ctx_unique should have the same length as the original context
len(ctx_unique)

41

# Chain

In [143]:
answer = []
context = []

In [144]:
import warnings
warnings.filterwarnings("ignore")

In [145]:
i = 1

for question in questions:
    try:
        result = chain_quality.invoke({"question":question, "chat_history":"-"})
        print("Successfully run the chain for question", i)
        answer.append(result['answer'])
        context.append(result['context'])
        i += 1
    
    except Exception:
        print("Failed run the chain for question", i)
        break



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (kl:Peraturan {jenis_peraturan: 'KEARIFAN LOKAL'})
MATCH (kl)-[:MENCABUT|MENGUBAH|DIUBAH|DICABUT]->(revoked:Peraturan)
RETURN 
  kl AS challengedRegulation, 
  revoked AS challengingRegulation

Full Context:
[]

> Finished chain.
Successfully run the chain for question 1


> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (lps:Peraturan {nomor: '4', tahun: '2023', jenis_peraturan: 'UNDANG-UNDANG', tentang: 'Lembaga Penjamin Simpanan'}),
      (otj:Peraturan {nomor: 'XXX', tahun: 'YYYY', jenis_peraturan: 'UNDANG-UNDANG', tentang: 'Otoritas Jasa Keuangan'})
MATCH (otj)-[:MENCABUT|MENGUBAH|DICABUT|DIUBAH]->(revoked:Peraturan)
WHERE revoked.tentang IN ['bank', 'asuransi']
RETURN 
  lps AS regulation, 
  otj AS revokedBy, 
  revoked AS revokedRegulation

Full Context:
[]

> Finished chain.
Successfully run the chain for question 2
Successfully run the chain for question 3
Successfully run the

In [146]:
len(context)

30

In [147]:
context_unique = unique_context(context)
# context_unique

In [148]:
ground_truth = list(dataset2["ground_truth"])
ground_truth

['The concept of kearifan lokal in Bali faces challenges such as a decline in the number and quality of institutions, facilities, human resources, and the value systems and cultural institutions. Additionally, the development in Bali has not fully guaranteed the preservation of cultural values, traditions, and local wisdom as the identity of the Balinese people.',
 'Lembaga Penjamin Simpanan (LPS) has specific responsibilities under Undang-undang Nomor 4 Tahun 2023 regarding the management and resolution of banks and insurance companies that have had their licenses revoked by Otoritas Jasa Keuangan, which include: 1. Managing and handling the Bank Dalam Resolusi and insurance companies that are being addressed and resolved by LPS; 2. Managing the assets of the Bank Dalam Resolusi and insurance companies that are handed over for handling or resolution to LPS; and 3. Settling the assets and liabilities of the Bank Dalam Resolusi and insurance companies whose licenses have been revoked by

In [149]:
# context_unique_str = []

# for i in context_unique:
#     ls = []
#     for j in i:
#         ls.append(j.page_content)
#     context_unique_str.append(ls)
    
context_unique_str = ['\n'.join(ctx) if isinstance(ctx, list) else str(ctx) for ctx in context_unique]
context_as_list = [[ctx] if isinstance(ctx, str) else ctx for ctx in context_unique]

In [150]:
len(context_unique)

30

In [151]:
len(context_unique_str[0])

12859

In [152]:
from datasets import Dataset

dict_eval = {
    "question": questions,
    "answer": answer,
    "contexts": context_as_list,
    "ground_truth": ground_truth
}

dataset = Dataset.from_dict(dict_eval)

In [153]:
import nest_asyncio
import warnings

nest_asyncio.apply()
warnings.filterwarnings("ignore")

In [154]:
from utils.llm import get_llm, get_embedding_model

llm = get_llm("nvidia-llama3.1")
embed_model = get_embedding_model("BAAI/bge-m3")

In [155]:
from ragas.metrics import (FaithulnesswithHHEM, answer_relevancy, context_precision, context_recall, context_utilization, context_entity_recall,
noise_sensitivity_relevant, noise_sensitivity_irrelevant, answer_similarity, answer_correctness, faithfulness)
from ragas import evaluate

# faithfulness_with_hhem = FaithulnesswithHHEM()

metrics = [answer_relevancy, context_precision, context_recall, context_utilization, context_entity_recall,
           answer_similarity, answer_correctness, faithfulness]

In [156]:
score = evaluate(dataset, metrics, llm, embed_model)

Evaluating:   0%|          | 0/240 [00:00<?, ?it/s]

Exception raised in Job[132]: TimeoutError()
Exception raised in Job[204]: TimeoutError()
Exception raised in Job[180]: TimeoutError()
Exception raised in Job[124]: TimeoutError()
Exception raised in Job[148]: TimeoutError()
Exception raised in Job[52]: TimeoutError()
Exception raised in Job[212]: TimeoutError()
Exception raised in Job[100]: TimeoutError()
Exception raised in Job[60]: TimeoutError()
Exception raised in Job[4]: TimeoutError()
Exception raised in Job[220]: TimeoutError()
Exception raised in Job[76]: TimeoutError()
Exception raised in Job[228]: TimeoutError()
Exception raised in Job[20]: TimeoutError()
Exception raised in Job[196]: TimeoutError()
Exception raised in Job[236]: TimeoutError()
Exception raised in Job[188]: TimeoutError()
Exception raised in Job[164]: TimeoutError()
Exception raised in Job[116]: TimeoutError()
Exception raised in Job[68]: TimeoutError()
Exception raised in Job[44]: TimeoutError()
Exception raised in Job[108]: TimeoutError()
Exception raised i

In [157]:
score

{'answer_relevancy': 0.7111, 'context_precision': 0.8817, 'context_recall': 0.8868, 'context_utilization': 0.9539, 'context_entity_recall': 0.2917, 'answer_similarity': 0.7527, 'answer_correctness': 0.5728, 'faithfulness': 0.7811}

In [ ]:
#